**Parcial 2 : Análisis de datos del mercado laboral GEIH 2026: DESARROLLO**

1.Para el total nacional calcule (a Junio de 2026): 
- población en edad de trabajar  
- población que no está en edad de trabajar 
- población económicamente activa 
- población económicamente inactiva 
- tasa de ocupación -tasa de desempleo  

In [0]:
import pandas as pd

In [0]:
# Seleccionar el período de referencia
fuerzadetrabajo_junio = fuerzadetrabajo[fuerzadetrabajo["PERIODO"] == 20260626].copy()

In [0]:
fuerzadetrabajo_junio

In [0]:
# Calcular la población en edad de trabajar
PET = fuerzadetrabajo_junio.loc[
    fuerzadetrabajo_junio["PET"] == 1, "FEX_C18"
].sum()

print(f"Población en edad de trabajar: {PET:,.0f} personas")


In [0]:
# Calcular la PET desde características generales
PET_CGSE = CGSE.loc[
    (CGSE["PERIODO"] == 20260626) &
    (CGSE["P6040"] >= 15), "FEX_C18"
].sum()

print(f"PET desde Fuerza de trabajo: {PET:,.0f}")
print(f"PET desde Características generales: {PET_CGSE:,.0f}")

**_R/ el total nacional de personas en edad de trabajar para junio de 2026 es de 24,095,346 personas _**

In [0]:
# Filtrar el período de junio de 2026
CGSE = spark.read.format("delta").table("workspace.electiva.CGSE").toPandas()
CGSE_junio = CGSE.loc[
    CGSE["PERIODO"] == 20260626
].copy()

In [0]:
CGSE_junio

In [0]:
# Calcular la población que no está en edad de trabajar
# En este cálculo: personas menores de 15 años
poblacion_no_PET = CGSE_junio.loc[
    CGSE_junio["P6040"] < 15, "FEX_C18"
].sum()

print(
    f"Población que no está en edad de trabajar: "
    f"{poblacion_no_PET:,.0f} personas"
)

**_**R/ El total nacional de personas que no esta en edad de trabajar para junio de 2026 es de 7,275,710 personas ``**_**

In [0]:
# Población económicamente activa (PEA)
PEA = fuerzadetrabajo_junio.loc[
    fuerzadetrabajo_junio["FT"] == 1, "FEX_C18"
].sum()

print("Población económicamente activa:", round(PEA))


**_**R/ El total nacional de la poblacion economicamente activa para junio de 2026 es de 15,432,739 personas ``**_**

In [0]:
# Población económicamente inactiva (PEI)
PEI = fuerzadetrabajo_junio.loc[
    fuerzadetrabajo_junio["FFT"] == 1, "FEX_C18"
].sum()

print("Población económicamente inactiva:", round(PEI))


**_**R/ El total nacional de la poblacion economicamente inactiva para junio de 2026 es de 8,662,607 personas ``**_**

In [0]:
# Filtrar el período de junio de 2026
ocupados = spark.read.format("delta").table("workspace.electiva.ocupados").toPandas()
ocupados_junio = ocupados.loc[
    ocupados["PERIODO"] == 20260626
].copy()

In [0]:
ocupados_junio

In [0]:
# Cargar y filtrar fuerza de trabajo para junio de 2026
fuerzadetrabajo = spark.read.format("delta").table("workspace.electiva.fuerzadetrabajo").toPandas()
fuerzadetrabajo_junio = fuerzadetrabajo[fuerzadetrabajo["PERIODO"] == 20260626].copy()

# Población en edad de trabajar (PET)
pet = fuerzadetrabajo_junio.loc[
    fuerzadetrabajo_junio["PET"] == 1, "FEX_C18"
].sum()

# Población ocupada (PO)
po = ocupados_junio["FEX_C18"].sum()

# Tasa de ocupación (TO)
tasa_ocupacion = (po / pet) * 100

print(f"Población en edad de trabajar: {pet:,.0f}")
print(f"Población ocupada: {po:,.0f}")
print(f"Tasa de ocupación: {tasa_ocupacion:.2f}%")

_**R/ La tasa de ocupacion nacional para el periodo dejunio 2026 es de 59,10%**_

In [0]:
# Cargar la tabla de desempleados y filtrar junio de 2026
desempleados = spark.read.format("delta").table("workspace.electiva.desempleados").toPandas()
desempleados_junio = desempleados[desempleados["PERIODO"] == 20260626].copy()

# Población desocupada (PD)
PD = desempleados_junio.loc[desempleados_junio["DSI"] == 1, "FEX_C18"].sum()

# Tasa de desempleo (TD)
tasa_desempleo = (PD / PEA) * 100

print(f"Población desocupada: {PD:,.0f}")
print(f"Tasa de desempleo: {tasa_desempleo:.2f}%")

_**R/ La tasa de desempleo nacional para el periodo dejunio 2026 es de 7,72%**_

2. Calcule las tasas de desempleo para hombres y mujeres en todos los departamentos de Colombia a Junio de 2026, ¿cuáles son los departamentos con mayores y menores tasas de desempleo?

In [0]:
import pandas as pd

In [0]:
#Leyendo la tabla de caracteristicas generales
ft = spark.read.format("delta").table("workspace.electiva.fuerzadetrabajo")
desempleados = spark.read.format("delta").table("workspace.electiva.desempleados")
cgse = spark.read.format("delta").table("workspace.electiva.cgse")

# Convertir el DataFrame de Spark a Pandas
ft = ft.toPandas()
desempleados = desempleados.toPandas()
cgse = cgse.toPandas()

#Primero hay que anexar la p3271 a la tabla de desempleados
desempleados = pd.merge(desempleados, cgse[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P3271','P6040']], 
                        on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'], how='left')

ft = pd.merge(ft, cgse[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P3271','P6040']], 
                        on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'], how='left')


In [0]:
desempleados

In [0]:
# desempleados_dpto -> cantidad de hombres y mujeres desempleados por departamento
desempleados_dpto_genero = desempleados[desempleados['DSI']==1].groupby(['P3271','DPTO'])['FEX_C18'].sum()

In [0]:
desempleados_dpto_genero

In [0]:
#cantidad de hombres y mujeres economicamente activos por departamento
pea_dpto_genero = ft[ft['FT']==1].groupby(['P3271','DPTO'])['FEX_C18'].sum()

In [0]:
pea_dpto_genero

In [0]:
tabla_desempleo = pd.DataFrame({'PEA': pea_dpto_genero, 'DESEMPLEADOS':desempleados_dpto_genero})

In [0]:
tabla_desempleo

In [0]:
tabla_desempleo['TD']= (tabla_desempleo['DESEMPLEADOS']/tabla_desempleo['PEA'])*100


In [0]:
tabla_desempleo.head(100)

In [0]:
# 4. Diccionarios de mapeo
dpto_nombre = {
    5: 'ANTIOQUIA', 8: 'ATLANTICO', 11: 'BOGOTA', 13: 'BOLIVAR', 15: 'BOYACA',
    17: 'CALDAS', 18: 'CAQUETA', 19: 'CAUCA', 20: 'CESAR', 23: 'CORDOBA',
    25: 'CUNDINAMARCA', 27: 'CHOCO', 41: 'HUILA', 44: 'GUAJIRA', 47: 'MAGDALENA',
    50: 'META', 52: 'NARIÑO', 54: 'NORTE DE SANTANDER', 63: 'QUINDIO', 66: 'RISARALDA',
    68: 'SANTANDER', 70: 'SUCRE', 73: 'TOLIMA', 76: 'VALLE DEL CAUCA', 81: 'ARAUCA',
    85: 'CASANARE', 86: 'PUTUMAYO', 88: 'SAN ANDRES', 91: 'AMAZONAS', 94: 'GUAINIA',
    95: 'GUAVIARE', 97: 'VAUPES', 99: 'VICHADA'
}

sexo_nombre = {1:'Hombre',2:'Mujer'}

tabla_desempleo = tabla_desempleo.reset_index()

tabla_desempleo['Departamento'] = tabla_desempleo['DPTO'].map(dpto_nombre)
tabla_desempleo['Sexo'] = tabla_desempleo['P3271'].map(sexo_nombre)


In [0]:
tabla_desempleo = tabla_desempleo[['Departamento','Sexo','PEA','DESEMPLEADOS','TD']]
tabla_desempleo

In [0]:
# Ordenar por dos variables de mayor a menor (descendente)
tabla_desempleo = tabla_desempleo.sort_values(by=[ 'Sexo','TD'], ascending=False)
tabla_desempleo

In [0]:
tabla_desempleo[tabla_desempleo['Sexo']=='Hombre']

_**R/ los departamentos con mayores tasas de desempleo son:
putumayo para las mujeres y san andres para loshombres 

y los departamentos con menores tasas de desempleo son:
bolivar para las mujeres y vaupes para los hombres**_
